# 01 - Exploratory Data Analysis: Fashion-MNIST

**Project:** DNN-Image-Classifier (fully connected network, no CNNs)

This notebook explores the Fashion-MNIST dataset before any modelling. The goals are to:

1. Load the dataset and inspect its structure
2. Analyse the class distribution and check for class imbalance
3. Visually inspect sample images
4. Compute basic pixel statistics
5. Save all key figures to `../reports/figures/`

The findings here directly inform the preprocessing choices (normalization, flattening) used later in `src/preprocessing.py`.

> **About the data:** Fashion-MNIST contains 70,000 grayscale images (28x28 px) of Zalando clothing articles across 10 classes: 60,000 for training and 10,000 for testing.

## 1. Setup

Imports, reproducibility seed, output folder, and plotting defaults.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from scipy import stats
from tensorflow.keras.datasets import fashion_mnist

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Output directory for figures (created if missing)
FIGURES_DIR = Path("../reports/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Plot styling
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight"})

# Fashion-MNIST label names (index = class id)
CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]


def save_figure(fig, filename):
    """Save a matplotlib figure to FIGURES_DIR and report the path."""
    path = FIGURES_DIR / filename
    fig.savefig(path)
    print(f"Saved: {path}")


print(f"TensorFlow version : {tf.__version__}")
print(f"Figures directory  : {FIGURES_DIR.resolve()}")

## 2. Loading the Dataset

`fashion_mnist.load_data()` downloads the dataset automatically on first use and caches it under `~/.keras/datasets/`, so later runs load instantly from disk.

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
print("Dataset loaded successfully.")

## 3. Data Shapes and Types

In [ ]:
print("TRAIN")
print(f"  Images : {x_train.shape}  dtype={x_train.dtype}")
print(f"  Labels : {y_train.shape}  dtype={y_train.dtype}")
print("\nTEST")
print(f"  Images : {x_test.shape}  dtype={x_test.dtype}")
print(f"  Labels : {y_test.shape}  dtype={y_test.dtype}")

n_train, n_test = len(x_train), len(x_test)
print(f"\nTotal samples : {n_train + n_test:,}  (train {n_train / (n_train + n_test):.1%} / test {n_test / (n_train + n_test):.1%})")
print(f"Flattened input size for the DNN : {x_train.shape[1] * x_train.shape[2]} features")

Each image is a 28x28 array of `uint8` values in `[0, 255]`. Because our model uses **fully connected layers only**, every image will later be flattened into a 784-dimensional vector.

In [ ]:
# Sanity checks: no missing values, labels within the expected range
assert x_train.shape[1:] == (28, 28) and x_test.shape[1:] == (28, 28)
assert not np.isnan(x_train).any() and not np.isnan(x_test).any()
assert set(np.unique(y_train)) == set(range(10)) == set(np.unique(y_test))
print("All sanity checks passed: correct shapes, no NaNs, labels 0-9 present in both splits.")

## 4. Class Distribution Analysis

Counting samples per class in the training and test sets.

In [ ]:
dist_df = pd.DataFrame({
    "class_id": range(10),
    "class_name": CLASS_NAMES,
    "train_count": np.bincount(y_train, minlength=10),
    "test_count": np.bincount(y_test, minlength=10),
})
dist_df["train_pct"] = (dist_df["train_count"] / dist_df["train_count"].sum() * 100).round(2)
dist_df["test_pct"] = (dist_df["test_count"] / dist_df["test_count"].sum() * 100).round(2)

dist_df

In [ ]:
colors = sns.color_palette("viridis", n_colors=10)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

for ax, column, title in zip(
    axes, ["train_count", "test_count"], ["Training set", "Test set"]
):
    bars = ax.bar(dist_df["class_name"], dist_df[column], color=colors, edgecolor="black", linewidth=0.5)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title(f"Class distribution - {title}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of samples")
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylim(0, dist_df[column].max() * 1.12)

fig.tight_layout()
save_figure(fig, "01_class_distribution.png")
plt.show()

## 5. Sample Images

A random selection of 20 training images with their labels (fixed seed, so the grid is reproducible).

In [ ]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(x_train), size=20, replace=False)

fig, axes = plt.subplots(4, 5, figsize=(11, 9.5))
for ax, idx in zip(axes.ravel(), sample_idx):
    ax.imshow(x_train[idx], cmap="gray")
    ax.set_title(f"{CLASS_NAMES[y_train[idx]]} ({y_train[idx]})", fontsize=10)
    ax.axis("off")

fig.suptitle("20 random training samples", fontsize=15, fontweight="bold")
fig.tight_layout()
save_figure(fig, "02_sample_images.png")
plt.show()

It is also useful to see one example per class side by side, which highlights which categories look alike (e.g. *Shirt*, *T-shirt/top*, *Pullover*, and *Coat*). A fully connected network has no built-in notion of spatial structure, so these visually similar classes are where we expect most of the errors.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for class_id, ax in enumerate(axes.ravel()):
    first_idx = np.where(y_train == class_id)[0][0]
    ax.imshow(x_train[first_idx], cmap="gray")
    ax.set_title(CLASS_NAMES[class_id], fontsize=11)
    ax.axis("off")

fig.suptitle("One example per class", fontsize=15, fontweight="bold")
fig.tight_layout()
save_figure(fig, "03_one_example_per_class.png")
plt.show()

## 6. Basic Pixel Statistics

Min, max, mean, and standard deviation of the raw pixel values.

In [ ]:
def pixel_stats(images):
    return {
        "min": int(images.min()),
        "max": int(images.max()),
        "mean": float(images.mean()),
        "std": float(images.std()),
    }


stats_df = pd.DataFrame({"train": pixel_stats(x_train), "test": pixel_stats(x_test)}).round(3)
stats_df

In [ ]:
zero_fraction = (x_train == 0).mean()
print(f"Fraction of pixels equal to 0 (background) in train set: {zero_fraction:.1%}")

# Statistics after scaling to [0, 1] - this is what the network will see
x_train_scaled = x_train.astype("float32") / 255.0
print(f"\nAfter dividing by 255:  mean={x_train_scaled.mean():.4f}  std={x_train_scaled.std():.4f}")

In [ ]:
# Histogram of pixel intensities (bincount is much faster than plotting 47M values)
pixel_counts = np.bincount(x_train.ravel(), minlength=256)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].bar(range(256), pixel_counts, width=1.0, color="steelblue")
axes[0].set_title("Pixel intensity histogram (training set)", fontweight="bold")
axes[0].set_xlabel("Pixel value")
axes[0].set_ylabel("Count")

axes[1].bar(range(256), pixel_counts, width=1.0, color="steelblue")
axes[1].set_yscale("log")
axes[1].set_title("Same histogram, log scale", fontweight="bold")
axes[1].set_xlabel("Pixel value")
axes[1].set_ylabel("Count (log)")

fig.tight_layout()
save_figure(fig, "04_pixel_intensity_histogram.png")
plt.show()

In [ ]:
# Mean pixel intensity per class: gives a rough feel for how much of the frame each item fills
per_class_mean = (
    pd.DataFrame({"class_name": [CLASS_NAMES[i] for i in y_train], "mean_pixel": x_train.reshape(len(x_train), -1).mean(axis=1)})
    .groupby("class_name")["mean_pixel"]
    .mean()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(per_class_mean.index, per_class_mean.values, color=sns.color_palette("mako", n_colors=10))
ax.set_title("Mean pixel intensity per class (training set)", fontweight="bold")
ax.set_xlabel("Mean pixel value (0-255)")
fig.tight_layout()
save_figure(fig, "05_mean_intensity_per_class.png")
plt.show()

**Takeaways:** most pixels are background (value 0), the intensity distribution is heavily skewed, and the raw 0-255 range is not suitable as direct network input. Scaling to `[0, 1]` (and optionally standardizing) will be part of preprocessing.

## 7. Class Imbalance Check

We quantify balance using:

- the **imbalance ratio** (largest class / smallest class; 1.0 means perfectly balanced), and
- a **chi-square goodness-of-fit test** against a uniform distribution (a high p-value means we cannot reject uniformity).

In [ ]:
def imbalance_report(counts, split_name):
    counts = np.asarray(counts)
    ratio = counts.max() / counts.min()
    chi2, p_value = stats.chisquare(counts)  # default expected = uniform
    print(f"{split_name}")
    print(f"  Largest class  : {counts.max():,}")
    print(f"  Smallest class : {counts.min():,}")
    print(f"  Imbalance ratio: {ratio:.2f}")
    print(f"  Chi-square     : {chi2:.2f}  (p-value = {p_value:.4f})")
    return ratio


train_ratio = imbalance_report(dist_df["train_count"], "TRAIN")
print()
test_ratio = imbalance_report(dist_df["test_count"], "TEST")

IMBALANCE_THRESHOLD = 1.5  # common rule of thumb for "noticeably imbalanced"
is_balanced = max(train_ratio, test_ratio) < IMBALANCE_THRESHOLD
print(f"\nVerdict: dataset is {'BALANCED' if is_balanced else 'IMBALANCED'} (threshold = {IMBALANCE_THRESHOLD}).")

In [ ]:
# Compare each class against the perfectly uniform expectation
uniform_share = 100 / 10
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(dist_df["class_name"], dist_df["train_pct"], color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(uniform_share, color="red", linestyle="--", label=f"Uniform expectation ({uniform_share:.0f}%)")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax.set_ylim(0, uniform_share * 1.4)
ax.set_title("Training class share vs. uniform expectation", fontweight="bold")
ax.set_ylabel("Share of training set (%)")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
save_figure(fig, "06_class_balance_check.png")
plt.show()

## 8. Saved Figures

All figures above were written to `../reports/figures/` (300 DPI), ready to embed in the project README.

In [ ]:
saved = sorted(FIGURES_DIR.glob("0*.png"))
print(f"{len(saved)} figures saved in {FIGURES_DIR.resolve()}:\n")
for path in saved:
    print(f"  - {path.name}  ({path.stat().st_size / 1024:.0f} KB)")

## 9. Summary and Next Steps

### Key findings

| Aspect | Finding |
|---|---|
| Size | 60,000 train / 10,000 test images, 28x28 grayscale |
| Data quality | No missing values; labels 0-9 present in both splits |
| Class balance | Perfectly balanced (6,000 per class in train, 1,000 in test) |
| Pixel range | Raw `uint8` 0-255, dominated by background (zeros) |
| Hard classes | Shirt, T-shirt/top, Pullover, Coat look visually similar |

### Implications for modelling

- **No resampling or class weights needed**, since the classes are balanced and plain accuracy is a reliable metric.
- **Normalize** pixels to `[0, 1]` (divide by 255) before training.
- **Flatten** each image to a 784-d vector for the fully connected network.
- Create a **validation split** from the training set (stratified) for early stopping and tuning.
- Expect most confusion among upper-body garments; the confusion matrix in `evaluate.py` should be inspected for this.

**Next:** implement `src/data_loader.py` and `src/preprocessing.py`.